In [1]:
# lets just look at the senseval2 format data to see if we can get it and put it in a nice easy file

# then we want to prepare the datasets that we'll use for the semcor/pedersen validation experiment.

In [2]:
import nltk
import pandas as pd

In [3]:
fileids = [
    'hard.pos',
    'line.pos',
    'interest.pos',
    'serve.pos',
]
root = '/home/gsc685/data/pedersen_sense_tagged/'

In [4]:
corpus = nltk.corpus.reader.senseval.SensevalCorpusReader(root, fileids=fileids)

In [5]:
instances = corpus.instances()

In [6]:
instances[0]

SensevalInstance(word='hard-a', position=20, context=[('``', '``'), ('he', 'PRP'), ('may', 'MD'), ('lose', 'VB'), ('all', 'DT'), ('popular', 'JJ'), ('support', 'NN'), (',', ','), ('but', 'CC'), ('someone', 'NN'), ('has', 'VBZ'), ('to', 'TO'), ('kill', 'VB'), ('him', 'PRP'), ('to', 'TO'), ('defeat', 'VB'), ('him', 'PRP'), ('and', 'CC'), ('that', 'DT'), ("'s", 'VBZ'), ('hard', 'JJ'), ('to', 'TO'), ('do', 'VB'), ('.', '.'), ("''", "''")], senses=('HARD1',))

In [7]:
words = ' '.join([word for (word, pos) in instances[0].context])
words

"`` he may lose all popular support , but someone has to kill him to defeat him and that 's hard to do . ''"

In [8]:
# you want data in this format: word | word_form | sentence | pos | position | sense

def get_data(instances):
    data = []
    for instance in instances:
        try:
            word = instance.word
            word_form = instance.context[instance.position][0]
            try:
                sentence = ' '.join([word for (word, pos) in instance.context])
            except:
                stripped_instance = [item for item in instance.context if isinstance(item, tuple)]
                sentence = ' '.join([word for (word, pos) in stripped_instance])
            pos = instance.context[instance.position][1]
            position = instance.position
            sense = instance.senses[0]
            data.append((word, word_form, sentence, pos, position, sense))
        except Exception as e:
            print(f"Error processing instance: {e}")
            continue
    return data

data = get_data(instances)
data[0]

('hard-a',
 'hard',
 "`` he may lose all popular support , but someone has to kill him to defeat him and that 's hard to do . ''",
 'JJ',
 20,
 'HARD1')

In [9]:
len(data)

15225

In [10]:
df = pd.DataFrame(data, columns=['word', 'word_form', 'sentence', 'pos', 'position', 'sense'])
# Remove the last two characters from the 'word' column
df['lemma'] = df['word'].str[:-2]

In [11]:
df.to_csv('/home/gsc685/data/collected_tokens/senseval2/pedersen_tokens.csv', index=True, index_label='sentence_id')

In [12]:
# save each word form in a separate file
for word in df['lemma'].unique():
    word_df = df[df['lemma'] == word]
    word_df.to_csv(f'/home/gsc685/data/collected_tokens/senseval2/{word}.csv', index=True, index_label='sentence_id')

In [13]:
df

,word,word_form,sentence,pos,position,sense,lemma
0,hard-a,hard,"`` he may lose all popular support , but someo...",JJ,20,HARD1,hard
1,hard-a,hard,clever white house `` spin doctors '' are havi...,JJ,10,HARD1,hard
2,hard-a,hard,i find it hard to believe that the sacramento ...,JJ,3,HARD1,hard
3,hard-a,hard,now when you get bad credit data or are confus...,JJ,15,HARD1,hard
4,hard-a,harder,'a great share of responsibility for this nati...,JJ,66,HARD1,hard
...,...,...,...,...,...,...,...
15220,serve-v,serving,piedmont 's baltimore hub and usair 's hub in ...,VBG,54,SERVE6,serve
15221,serve-v,serving,it isn 't known whether the irs will try to co...,VBG,33,SERVE6,serve
15222,serve-v,serving,"the upshot , for all practical purposes , is n...",VBG,23,SERVE6,serve
15223,serve-v,serve,the company said it is also adding executives ...,VB,47,SERVE6,serve


In [14]:
df.groupby('word').sense.value_counts()

word        sense     
hard-a      HARD1         3455
            HARD2          502
            HARD3          376
interest-n  interest_6    1252
            interest_5     500
            interest_1     361
            interest_4     178
            interest_3      66
            interest_2      11
line-n      product       2217
            phone          429
            text           404
            division       374
            cord           373
            formation      349
serve-v     SERVE10       1814
            SERVE12       1272
            SERVE2         853
            SERVE6         439
Name: count, dtype: int64

In [15]:
# lets see how many senses and instances there are for the words in semcor

In [16]:
semcor = pd.read_csv('/home/gsc685/data/semcor_all_tokens.csv', index_col=0)
semcor.head()


,lemma,sense,word_form,pos,sentence,category,fileid,domain
id,,,,,,,,
0,fulton county grand jury,Lemma('group.n.01.group'),fulton,NNP,The Fulton County Grand Jury said Friday an in...,news,brown1/tagfiles/br-a01.xml,noun.Tops
1,fulton county grand jury,Lemma('group.n.01.group'),county,NNP,The Fulton County Grand Jury said Friday an in...,news,brown1/tagfiles/br-a01.xml,noun.Tops
2,fulton county grand jury,Lemma('group.n.01.group'),grand,NNP,The Fulton County Grand Jury said Friday an in...,news,brown1/tagfiles/br-a01.xml,noun.Tops
3,fulton county grand jury,Lemma('group.n.01.group'),jury,NNP,The Fulton County Grand Jury said Friday an in...,news,brown1/tagfiles/br-a01.xml,noun.Tops
4,said,Lemma('state.v.01.say'),said,VB,The Fulton County Grand Jury said Friday an in...,news,brown1/tagfiles/br-a01.xml,verb.communication


In [17]:
lemmas = ['line', 'hard', 'interest', 'serve']
semcor_lemmas = semcor[semcor.lemma.isin(lemmas)]


In [18]:
semcor_lemmas.groupby('category').sense.value_counts()

category         sense                       
adventure        Lemma('difficult.a.01.hard')    2
                 Lemma('hard.a.02.hard')         2
                 Lemma('hard.r.02.hard')         2
                 Lemma('line.n.14.line')         2
                 Lemma('hard.r.01.hard')         1
                                                ..
romance          Lemma('line.n.04.line')         1
                 Lemma('serve.v.01.serve')       1
                 Lemma('serve.v.05.serve')       1
science_fiction  Lemma('serve.v.01.serve')       1
                 Lemma('serve.v.02.serve')       1
Name: count, Length: 178, dtype: int64

In [19]:
semcor_lemmas.groupby('lemma').sense.value_counts()

lemma     sense                            
hard      Lemma('difficult.a.01.hard')         23
          Lemma('hard.a.03.hard')              14
          Lemma('hard.a.02.hard')              13
          Lemma('hard.r.01.hard')              10
          Lemma('arduous.s.01.hard')            3
          Lemma('hard.r.02.hard')               2
          Lemma('hard.s.04.hard')               2
          Lemma('hard.r.03.hard')               1
          Lemma('hard.r.04.hard')               1
          Lemma('hard.r.05.hard')               1
          Lemma('person.n.01.person')           1
interest  Lemma('interest.n.01.interest')      57
          Lemma('sake.n.01.interest')          32
          Lemma('interest.n.03.interest')      20
          Lemma('interest.n.04.interest')      14
          Lemma('interest.n.05.interest')       7
          Lemma('interest.n.06.interest')       5
          Lemma('pastime.n.01.interest')        3
          Lemma('interest.v.01.interest')       2
line  

In [20]:
semcor_lemmas.to_csv('/home/gsc685/data/collected_tokens/semcor/pedersen_tokens.csv', index=True, index_label='id')